# Load Libraries

In [1]:
import torch
from torch.utils.data import DataLoader, Dataset
from torch import nn, optim
from torch.optim import lr_scheduler
from torch.cuda.amp import GradScaler, autocast

import numpy as np
import pandas as pd
import os

# Import Pre-trained Image Models (TIMM)
import timm

from util.Trainer import EyeDiseaseClassifier, train_model, evaluate_model

c:\Users\kanek\miniconda3\envs\PyTorch-DL\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Global Variables

In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 0.001
NUM_CLASSES = 8
IMG_SIZE = 224
NUM_WORKERS = 4
DROP_RATE = 0.5
WEIGHT_DECAY = 1e-4

Using device: cuda:0


# Load Data

In [3]:
#  Load Data from pt file
base_path = "./Data/Final_Dataset/"
train_ds = torch.load(base_path + "train_dataset.pt")
balanced_train_ds = torch.load(base_path + "balanced_train_dataset_dataset.pt")
test_ds = torch.load(base_path + "test_dataset.pt")
valid_ds = torch.load(base_path + "val_dataset.pt")
class_names = torch.load(base_path + "class_map.pt")

# Create DataLoaders

In [4]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
balanced_train_loader = DataLoader(balanced_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [5]:
original_dataloaders = {
    "train": train_loader,
    "valid": valid_loader,
    "test": test_loader
}

balanced_dataloaders = {
    "train": balanced_train_loader,
    "valid": valid_loader,
    "test": test_loader
}

# Create Model

In [6]:
model = EyeDiseaseClassifier(
        model_name="efficientnet_b0",
        num_classes=NUM_CLASSES,
        dropout_rate=DROP_RATE,
        in_channels=1
    )
model = model.to(device)

# Set Training Param

In [7]:
criterion = nn.CrossEntropyLoss()
    
# Define optimizer
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Define learning rate scheduler
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# Train Model

In [8]:
model, history = train_model(
        model=model,
        dataloaders={'train': train_loader, 'val': valid_loader},
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        num_epochs=NUM_EPOCHS,
    )

Epoch 1/10
----------


train:   2%|▏         | 2/84 [00:55<38:06, 27.88s/it]


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
results = evaluate_model(
        model=model,
        test_loader=test_loader,
        device=device,
        class_names=class_names
    )